# Start of the Gold process

This notebook initiates the transformation of the Gold layer, which receives data already loaded in the Silver layer, creating table aggregation for analysis.

- Definition of variables, paths, and catalog schema
- Configuration of credentials and secure access
- Create analytic Gold tables in catalog with SQL
- Writing the Gold tables to the Data Lake container

In [0]:
from datetime import datetime as dt
tier = "gold"
raw_data = 'olistdataraw'
storage_account = "databrickslrd"

spark.sql("CREATE SCHEMA IF NOT EXISTS " + tier)
spark.sql("USE " + tier)


access_key = dbutils.secrets.get(
    scope = "olist-scope", 
    key = "access-key"
    )
    
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    access_key
)

PATH_STORAGE = f"wasbs://{tier}@{storage_account}.blob.core.windows.net"

# Write function
def write_delta(table):
    df = spark.table(table)
    df.write.format("delta").mode("overwrite").save(f"{PATH_STORAGE}/{table}_{dt.now().strftime('%Y%m%d')}")
    return 

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;


CREATE OR REPLACE TABLE gold.agg_item_location
USING DELTA
AS

WITH geo AS (
  SELECT 
    CITY,
    AVG(LATITUDE) AS LATITUDE,
    AVG(LONGITUDE) AS LONGITUDE
  FROM silver.olist_geolocation
  GROUP BY CITY
)

SELECT
  o.ID_ORDER,
  geo.LATITUDE,
  geo.LONGITUDE

FROM silver.olist_orders o
JOIN silver.olist_customers s ON o.ID_CUSTOMER = s.ID_CUSTOMER
JOIN silver.olist_order_items i ON o.ID_ORDER = i.ID_ORDER
JOIN geo 
  ON s.CITY = geo.CITY

In [0]:

write_delta("agg_item_location")

In [0]:
%sql
CREATE OR REPLACE TABLE gold.agg_review_location
AS
SELECT  
  olist_products.PRODUCT_CATEGORY AS PRODUCT,
  review.REVIEW_SCORE AS SCORE,
  olist_customers.CITY
FROM silver.olist_order_reviews review
JOIN silver.olist_order_items item ON review.ID_ORDER = item.ID_ORDER
JOIN silver.olist_products ON olist_products.ID_PRODUCT = item.ID_PRODUCT
JOIN silver.olist_orders ON olist_orders.ID_ORDER = review.ID_ORDER
JOIN silver.olist_customers ON olist_customers.ID_CUSTOMER = olist_orders.ID_CUSTOMER


In [0]:
write_delta("agg_review_location")

In [0]:
%sql
CREATE OR REPLACE TABLE gold.agg_product_order
AS
SELECT
  o.ID_ORDER,
  p.PRODUCT_CATEGORY

FROM silver.olist_orders o
JOIN silver.olist_customers s ON o.ID_CUSTOMER = s.ID_CUSTOMER
JOIN silver.olist_order_items i ON o.ID_ORDER = i.ID_ORDER
JOIN silver.olist_products P ON i.ID_PRODUCT = P.ID_PRODUCT

In [0]:
write_delta("agg_product_order")